# SCiO: scan -> spectrum

Capture a scan from the SCiO over USB and turn it into a 331-point reflectance
spectrum. Every step is one call into the `scio` library (`src/scio/`); this
notebook only supplies the parameters and draws the picture.

**Capture and processing are separate.** A scan is written to
`01_rawdata/scans/` as a *self-contained* `scio-scan/2` record - the raw blobs,
the white reference, the device id and i2s tag, both timestamps, temperatures
and calibration state. No network is involved. The Consumer Physics server turns
that record into a spectrum whenever you next have an account and a connection:
today, or in a year, from a different machine. That is deliberate - the server is
currently the only way to decode a SCiO blob, and it may be switched off.

**Before you start:** the SCiO answers only when fully awake (steady blue). If it
pulses slowly it is idling: unplug, long-press off, long-press on, replug.

## 1. Annotate this scan

These four values go into the record. `SCAN_NAME` is mandatory - it is how you
will recognise this scan later. `SCAN_ID` is generated if you leave it empty.
The timestamp is taken automatically.

In [ ]:
SCAN_NAME = "my sample"        # what you are pointing at, e.g. "bark", "soil crust"
SCAN_ID   = ""                 # your own id; empty = generated
COMMENT   = ""                 # anything you will want to know a year from now

N_SCANS         = 1            # replicates to take in one go
FORCE_CALIBRATE = False        # True = take a fresh white reference first (cover on)

PORT = None                    # None = autodetect the SCiO's COM port

## 2. Library

`src/` holds the working pipeline (`scio`). Offline-decoding research lives in
`dev/` and is deliberately not imported here.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt

from scio import cloud, credentials, session, store, usb

SCANS_DIR     = store.SCANS_DIR        # 01_rawdata/scans      - raw, self-contained
PROCESSED_DIR = store.PROCESSED_DIR    # 02_processed_data     - records + spectra

print("raw scans      ->", SCANS_DIR)
print("processed data ->", PROCESSED_DIR)
print(session.summary())

## 3. Connect

The SCiO enumerates as a TI CDC serial port, VID:PID `0451:16AA`.

In [ ]:
ports = usb.find_scio_ports()
for p in ports:
    print(("* " if p["is_scio"] else "  ") + f'{p["device"]:8s} {p["vidpid"]}  {p["description"]}')

port = PORT or next((p["device"] for p in ports if p["is_scio"]), None)
if port is None:
    raise SystemExit("No SCiO found. Is it plugged in and steady blue?")
print("\nusing", port)

In [ ]:
dev = usb.ScioUSB(port).open()

info = dev.read_device_info()
for k in ("device_id", "firmware_version", "i2s_tag_config", "device_name", "ble_id", "dsp_id"):
    print(f"  {k:18s} {info.get(k)}")
if info.get("i2s_tag_missing"):
    print("\n  !! i2s_tag_config is empty - the server rejects scans without it.")
    print("     Power-cycle the device and re-run this cell.")

print("\n  temperature", dev.read_temperature())
print("  battery    ", dev.read_battery())

## 4. Capture

`session.capture` decides whether a white reference is needed, using the same
rules as the app (`NEVER` / `TIME_THRESHOLD` / `EXCEED_SCANS_LIMIT` /
`TEMP_THRESHOLD` / `NO_NEED`). The decision is **client-side**: the server never
asks for one and has never rejected a scan for a stale white reference. For this
device the server's thresholds are effectively infinite, so in practice a white
reference is taken only when none exists - or when you set `FORCE_CALIBRATE`.

If one is needed you will be asked to **put the SCiO in its box / cover on** and
press Enter.

In [ ]:
def ask_for_white_reference(report):
    print(f"White reference needed: {report['status']}")
    if report.get("wr_age_ms") is not None:
        print(f"  current WR is {report['wr_age_ms'] / 3.6e6:.1f} h old, "
              f"{report['scans_since_calibration']} scans since")
    if report.get("temp_delta") is not None:
        print(f"  temperature drift since the WR: {report['temp_delta']:.1f} degC")
    input("  >> Put the cover on / place the SCiO in its box, then press Enter ")
    return True


captured = []
for i in range(N_SCANS):
    name = SCAN_NAME if N_SCANS == 1 else f"{SCAN_NAME} {i + 1}"
    if N_SCANS > 1 and i:
        input(f"  >> Aim at the target for replicate {i + 1}/{N_SCANS} and press Enter ")
    path = session.capture(
        dev, name, SCAN_ID or None, COMMENT,
        out_dir=SCANS_DIR,
        force_calibrate=FORCE_CALIBRATE and i == 0,
        on_calibration_needed=ask_for_white_reference,
    )
    captured.append(path)
    rec = session.load_record(path)
    print(f"[{i + 1}/{N_SCANS}] {path.name}")
    print(f"        blobs {rec['raw']['sample']['size']}/{rec['raw']['sample_dark']['size']}"
          f"/{rec['raw'].get('sample_gradient', {}).get('size')} B, "
          f"WR {rec['white_reference']['sampled_white_at']}, "
          f"calibration {rec['calibration']['status_at_scan']}")

In [ ]:
dev.close()
print("device closed - the rest of this notebook needs no hardware")

## 5. Process

The record goes to `POST /v2/consumer/spectro-scan`, which returns the
331-point reflectance curve (740-1070 nm). Credentials are asked for **once** and
kept in an encrypted, gitignored file; access tokens are short-lived, so a fresh
one is fetched per request.

Result: `02_processed_data/<scan>_spectrum.json` (the whole scan record *plus*
the spectrum, so it too stands alone).

In [ ]:
token = credentials.get_token()      # prompts only if there is nothing stored, or it failed
print("logged in")

In [ ]:
processed = []
for path in captured:
    out = session.process(path, credentials.get_token(), out_dir=PROCESSED_DIR)
    processed.append(out)
    print("->", out.name)

In [ ]:
import json

fig, ax = plt.subplots(figsize=(9, 4.5))
for out in processed:
    d = json.loads(Path(out).read_text(encoding="utf-8"))
    s, a = d["spectrum"], d["annotation"]
    ax.plot(s["wavelength_nm"], s["reflectance"], lw=1.2,
            label=f"{a['name']} ({a['scan_id']})")
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("reflectance")
ax.set_title(SCAN_NAME)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Offline mode

Capture now, upload later. Run **sections 1-4 only** (no login) and the records
sit in `01_rawdata/scans/` until you come back to them. Nothing is lost: each
record already contains everything the server will ask for.

In [ ]:
todo = session.pending(SCANS_DIR, PROCESSED_DIR)
print(f"{len(todo)} scan(s) waiting to be processed")
for p in todo[:10]:
    print("  ", p.name)
if len(todo) > 10:
    print(f"   ... and {len(todo) - 10} more")

In [ ]:
# Upload the backlog. limit= keeps it small; pause= is polite to the server.
# Failures are recorded per scan and do not stop the run.
rows = session.process_pending(limit=2, pause=20.0,
                               scans_dir=SCANS_DIR, processed_dir=PROCESSED_DIR,
                               on_result=lambda r: print(
                                   ("OK   " if r["error"] is None else "FAIL ") + r["scan"]
                                   + ("" if r["error"] is None else "  " + r["error"][:80])))

## 7. What is in the store

The raw store also holds ~97 older captures converted into the same canonical
format: the 2020/2021 app-log scans (with the spectra the server returned at the
time, which double as a regression target), the 2023 USB series, and the 2026
series. The originals are untouched - these are additional copies.

In [ ]:
for k, v in session.summary(SCANS_DIR, PROCESSED_DIR).items():
    print(f"  {k:24s} {v}")